### Suchen auf Graphen, Tiefen- und Breitensuche
(auch genannt **DFS** (depth-first search) und **BFS** (breadth-first search).

Probleme die DFS löst:
- Zusammenhangskomponente finden,
- Weg durch Labyrinth finden,
- Sudoku lösen.


Probleme die BFS löst:
- Zusammenhangskomponente finden,
- kürzester Weg in Graph/Netzwerk finden,
- (kleine) Sokoban Probleme lösen.

Wir erläutern diese Suchstrategien anhand folgender Situation.
Ein Höhlenforscher hat die Aufgabe, ein grosses unterirdisches Höhlensystem zumindest zum Teil zu kartieren.
Das System besteht aus vielen Kammern,
welche durch kurze Tunnels verbunden sind. 
2 Kammern sind durch höchstens einen Tunnel verbunden. 
Jede Kammer ist mit einer eindeutigen Zahl markiert, und jeder Tunneleingang zeigt die Nummer der
Kammer zu der dieser führt (ein Tunnel ist eher eine Türe mit Zimmernummer).    
Der Forscher ist in einer ersten Kammer, von wo aus er seine Erkundung beginnt.
Er darf keine Markierungen am Höhlensystem anbringen, darf sich aber Notizen machen, will diese aber knapp halten.
Er verwaltet 

1. eine Liste  `zu_besuchende_Kammern` mit noch unbesuchten Kammern,
2. einen Dictionary `go_back`. Der Wert ist die Kammer(nummer), die der Tunnel verlässt, der
  Schlüssel ist die Kammer(nummer), wo der Tunnel hinführt. Der Dict `go_back` zeigt den Weg zurück zur Kammer, wo die Suche begann.

Zu Beginn, sieht sein Notizbuch so aus:  
-  `zu_besuchende_Kammern = [eine erste Türnummer]`  
-  `go_back = {eine erste Kammer: None}`  

Nun wiederholt er Folgendes, bis er einen Schatz gefunden (alle Schätze bez. Liste leer/alle Kammern besucht) hat.
  
- Er entfernt eine Türnummer aus seiner Liste und betritt diese Kammer.
- Er schaut reihum auf alle Türnummern.
  Ist die Türnummer noch kein Schlüssel im Dict, updatet er ihn mit
  > `{Nummer auf Tür: Kammernummer}`
  
  und fügt die Türnummer in die Liste an.
  
    Dass eine  Türnummer  bereits im Dict ist, bedeutet, dass er bereits in einer anderen Kammer war, von wo aus er diese Kammer betreten kann.
   
  

**Strategien**:
- depth-first:
  Gehe falls möglich immer von der aktuellen Kammer zu einer noch unbesuchten Kammer.
  Anderfalls entferne das letzte Element der Liste `zu_besuchende_Kammern` und setze die Erkundung von dort aus fort.

- breadth-first:
  arbeitet dich konzentrisch vor. Besuche zuerst
  alle unbesuchten Kammern mit Abstand 1 vom Start, dann
  alle unbesuchten Kammern mit Abstand 2 vom Start, u.s.w.
  Füge zu besuchende Kammern von links in  `zu_besuchende_Kammern` ein,
  entferne dann das letzte Element und setze die Erkundung von dieser Kammer aus fort.
  
    ```python
  [K_3, K_2, K_1]  # entferne K_1, füge unbesuchte Nachbarn von links her ein

  [K_13, K_12, K_11,  K_3, K_2]  # entferne K_2, füge unbesuchte Nachbarn von links her ein

  [K_23, K22, K21, K_13, K_12, K_11,  K_3]  # entferne K_3, füge unbesuchte Nachbarn von links her ein 
   
    ```
 


- smart guess (suchen  z.B. einen Ausgang)
  füge nicht nur die Türnummer in die Liste ein, sondern `(Stärke des Luftstroms, Türnummer)`
  (gehe kurz durch Tür, um  Luftstroms zu messsen).
  Entferne dann jeweils die Kammer mit dem grössten Luftstrom aus `zu_besuchende_Kammern` und setze die Erkundung von dieser Kammer aus fort.
  
  
**Hinweise zur Implementation dieser Strategien**
- depth-first: besuche immer die letzte Kammer in der Liste (`zu_besuchende_Kammern.pop()`).
  Hänge zu besuchende Kammern an Ende der Liste an.
- breadth-first: besuche immer die erste Kammer in der Liste (`zu_besuchende_Kammern.pop(0)`).
  Hänge zu besuchende Kammern an Ende der Liste an.
  Statt einer Liste benutzt man besser einen `deque` (double ended queue).
- smart guess: füge Tuple `(Präferenz, Kammernummer)` in die Liste ein.
  Entferne jeweils das Tuple mit der grössten Präferenz.
  Statt einer Liste benutzt man besser einen sog. `heap` (ein Heap ist eine Datanstruktur, aus der sich rasch das kleinste/grössten Element entfernen lässt).

Falls man  nur an der Nummer der Schatzkammer interessiert ist, nicht aber wie man von dort wieder heim findet,
genügt es, die besuchten Knoten in einer Menge `visited` oder `seen` zu speichern,
anstatt im Dict `go_back`.

  
**Terminology**:

Das Höhlensystem ist der Suchraum. Dieser Suchraum ist ein Graph, eine Menge von Knoten,
und zu jedem Knoten hat man eine Liste mit den direkten Nachbarknoten.

Einen Graphen kann man als Dict repräsentieren,
`graph = {node: neighbors,...}`. 
Manchmal hat man auch eine Menge mit den Knoten, und eine
Funktion `get_neigbors(node)`, welche die Nachbarknoten berechnet.

Eine Liste von Knoten [$k_1, k_2, \ldots, k_n$] mit der Eigenschaft, dass
jeweils $k_{n+1}$ Nachbar von $k_{n}$ ist, nennt man **Weg** oder **Pfad** von $k_1$ nach $k_n$.
Ist $k_1$ gleich $k_n$, so nennt man den Weg einen **Kreis**.

Der Dict `go_back` beschreibt einen **Baum**.
Ein Baum hat **genau einen Wurzelknoten**, ein Knoten ohne Vorgänger/Elternknoten (der Knoten `node` mit
`go_back[Knoten] is None`.
Jeder weitere Knoten hat genau einen Elternknoten, gegeben durch `go_back[Knoten]`.
Ein Knoten mit einem Elternknoten heisst **Kind/child** dieses Knotens.
Ein Knoten ohne Kinder heisst **Blatt/leaf**.
Ein **Baum ist immer kreisfrei**, es gibt keine geschlossenen Wege.

In [ ]:
from collections import deque


def search_df(node, get_neighbors, is_goal=None):
    nodes_to_visit = [node]
    go_back = {node: None}

    while nodes_to_visit:
        node = nodes_to_visit.pop()
        if is_goal and is_goal(node):
            break

        for neighbor in get_neighbors(node):
            if neighbor in go_back:
                continue

            go_back[neighbor] = node
            nodes_to_visit.append(neighbor)

    return node, go_back


def search_bf(node, get_options, is_goal):
    '''node: start node'''
    nodes_to_visit = deque([node])
    go_back = {node: None}

    while nodes_to_visit:
        node = nodes_to_visit.pop()
        if is_goal and is_goal(node):
            break

        for neighbor in get_neighbors(node):
            if neighbor in go_back:
                continue

            go_back[neighbor] = node
            nodes_to_visit.append(neighbor)

    return node, go_back


def path_home(node, go_back):
    path = []
    while node:
        path.append(node)
        node = go_back[node]

    return path

In [ ]:
import random
import matrix_helpers as M


def get_random_gird(size=8):
    matrix = M.make_matrix(size)
    for i in range(size**2):
        M.set_item(matrix, M.idx2pos(i, ncol=size), random.randint(0, 2))
    return matrix


def get_mines_grid(size=8, n_mines=10):
    '''liefert size x size Gitter mit 0 oder 1, 1 steht fuer eine Mine'''
    matrix = M.make_matrix(size, default=0)
    for i in random.sample(range(size**2), n_mines):
        M.set_item(matrix, M.idx2pos(i, ncol=size), 1)
    return matrix


def get_count_grid(mat):
    '''liefert Gitter mit dem Minencount'''
    ncol, nrow = M.get_dims(mat)
    count_grid = M.make_matrix(ncol, nrow)
    for pos in M.positions(ncol, nrow):
        count = sum(M.get_neighbor_vals(mat, pos, kinds='sc'))
        M.set_item(count_grid, pos, count)
    return count_grid


def get_neighbors(matrix, pos0):
    for pos in M.get_neighbors(matrix, pos0, kinds='sc'):
        if M.get_item(matrix, pos) == M.get_item(matrix, pos0):
            yield pos


def get_save_neighbors(matrix, pos0):
    if M.get_item(matrix, pos0) != 0:
        return

    yield from M.get_neighbors(matrix, pos0, kinds='sc')

    #  Kurzform fuer
    # for pos in M.get_neighbors(matrix, pos0, kinds='sc'):
    #     yield pos

In [ ]:
mat = get_mines_grid(5, 5)
mat

In [ ]:
matrix = get_count_grid(mat)
matrix

In [ ]:
_, go_back = search_df((4, 0), lambda pos: get_save_neighbors(matrix, pos))
set(go_back)

In [ ]:
matrix = get_random_gird()
M.show_matrix(matrix)

In [ ]:
_, go_back = search_df((0, 0), lambda pos: get_save_neighbors(matrix, pos))
set(go_back)

In [ ]:
list(get_save_neighbors(matrix, (1, 1)))

In [ ]:
path_home((5, 1), go_back)